# PID-RL Inference — Run Trained Model in Colab

This notebook runs the **trained PID-RL model** (Qwen 2.5 1.5B + LoRA) for inference and evaluation.

## What you need before starting:

1. **Base model**: `unsloth/Qwen2.5-1.5B-Instruct` (will be downloaded automatically)
2. **LoRA weights**: Download from [your HuggingFace repo or upload manually]

**Cells 1-2**: Environment setup  
**Cell 3**: Clone the repo (to get `eval.py`, `scenarios.py`, etc.)  
**Cell 4**: Load base model + LoRA adapter  
**Cell 5**: Run evaluation vs baselines (equivalent to Cell 11)  
**Cell 6**: Quick qualitative test (equivalent to Cell 12)

## 1 · Environment check

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2 · Install dependencies

In [ ]:
%%capture
!pip install --upgrade pip
!pip install unsloth vllm transformers peft accelerate
!pip install yfinance


## 3 · Clone the repo
Gets `eval.py`, `scenarios.py`, `sim.py`, `prompt.py`, `reward.py`, `baselines.py`.

In [ ]:
import os, sys, subprocess

REPO_DIR = "/content/qwen-rl-shanghai"
if not os.path.isdir(REPO_DIR):
    subprocess.run([
        "git", "clone",
        "https://github.com/Grinta-Protocol/qwen-rl-shanghai.git",
        REPO_DIR,
    ], check=True)

sys.path.insert(0, f"{REPO_DIR}/pid_rl")
os.chdir(f"{REPO_DIR}/pid_rl")
!ls

## 4 · Load base model + LoRA adapter

This cell downloads:
- Base model: `unsloth/Qwen2.5-1.5B-Instruct` (~3GB)
- LoRA weights: `Fenryr/qwen2.5-1.5B-pid-v1` (~37MB)


In [ ]:
from unsloth import FastLanguageModel
import torch

# === CONFIGURATION ===
# Your trained LoRA weights on HuggingFace
LORA_PATH = "Fenryr/qwen2.5-1.5B-pid-v1"

BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LEN = 2048

# === LOAD MODEL ===
print(f"Loading base model: {BASE_MODEL}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    fast_inference=False,
)

print(f"Loading LoRA adapter: {LORA_PATH}")
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=8,
)

from peft import PeftModel
model = PeftModel.from_pretrained(model, LORA_PATH)

FastLanguageModel.for_inference(model)
print("✓ Model loaded!")


## 5 · Evaluate vs baselines (Cell 11 equivalent)

Runs the trained model against baseline policies (Static, Heuristic, Random).

In [ ]:
import numpy as np
from config import SIM
from sim import Simulator
from scenarios import generate_batch
from prompt import build_prompt, parse_output, ParseError
from baselines import StaticGainsPolicy, HeuristicPolicy, RandomPolicy
from eval import run_full_eval


class TrainedModelPolicy:
    """Wraps the trained model to match the baselines.Policy interface."""
    name = "trained"

    def __init__(self, model, tokenizer, max_new_tokens=256, temperature=0.3):
        self.model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature

    def decide(self, state, btc_history, deviation_history):
        sys_p, user_p = build_prompt(
            state=state,
            btc_history=btc_history,
            deviation_history=deviation_history,
        )
        messages = [
            {"role": "system", "content": sys_p},
            {"role": "user",   "content": user_p},
        ]
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                temperature=self.temperature,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        gen = self.tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        return gen


trained_policy = TrainedModelPolicy(model, tokenizer)

policies = [
    StaticGainsPolicy(),
    HeuristicPolicy(),
    RandomPolicy(rng=np.random.default_rng(seed=123)),
    trained_policy,
]

print("Running full evaluation (synthetic holdout + real crashes)...")
results = run_full_eval(
    policies=policies,
    n_synthetic_holdout=24,
    include_real=True,
    verbose=False,
)

print("\n=== EVALUATION RESULTS ===")
for policy_name, metrics in results.items():
    print(f"\n{policy_name}:")
    for k, v in metrics.items():
        print(f"  {k}: {v}")

## 6 · Quick qualitative peek (Cell 12 equivalent)

Test the model on 3 scenarios: crash, pump, stable.

In [ ]:
from scenarios import _gen_crash, _gen_pump, _gen_stable

peek_rng = np.random.default_rng(seed=777)

for name, gen in [("crash", _gen_crash), ("pump", _gen_pump), ("stable", _gen_stable)]:
    sc = gen(peek_rng, SIM.episode_steps)
    sim = Simulator(
        btc_path=sc.btc_path, initial_kp=sc.initial_kp, initial_ki=sc.initial_ki,
        rng=np.random.default_rng(seed=7),
    )
    hist = sim.run_forward(n_steps=50)
    dev_hist = [h.deviation_pct for h in hist[-8:]]
    state = sim.get_state()
    completion = trained_policy.decide(
        state=state,
        btc_history=sc.btc_path[: sim.step_idx + 1],
        deviation_history=dev_hist,
    )
    parsed = parse_output(completion)
    print(f"\n=== {name} ===")
    print(f"  btc: {sc.btc_path[sim.step_idx - 5]:.0f} → {sc.btc_path[sim.step_idx]:.0f}")
    print(f"  dev: {state.deviation_pct:+.3f}%   kp={state.kp:.3f}  ki={state.ki:.5f}")
    print(f"  model raw: {completion[:300]}")
    print(f"  parsed:    {parsed}")